# Volatility Model Comparison

Standalone volatility-model and realized-volatility comparison extracted from Block 33 of `Options Pricing & Distribution.ipynb`. The `arch` package is required for the GARCH-family models.


In [ ]:
# Block 1: Standalone Setup and Underlying Data

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.io as pio

for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        PROJECT_ROOT = _project_root_candidate
        if str(PROJECT_ROOT) not in sys.path:
            sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise RuntimeError("Could not locate the Investment Research project root.")

SINGLE_ASSET_DIRECTORY = PROJECT_ROOT / "Research" / "Single Asset"
if str(SINGLE_ASSET_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SINGLE_ASSET_DIRECTORY))

from _params import get_single_asset_params
from Quantapp.data import get_market_history
from Quantapp.visualization.views.single_asset_profile.pricing.distribution import (
    plot_volatility_model_comparison_view,
)

pricing_params = get_single_asset_params()
ticker_str = pricing_params["ticker_str"]
interval = pricing_params["interval"]
period = pricing_params["period"]

TIMEFRAME_PROFILES = {
    "swing": {"short": 3, "mid": 9, "long": 21},
    "position": {"short": 21, "mid": 50, "long": 200},
    "structural": {"short": 200, "mid": 500, "long": 1000},
}
trading_strategy = "position"
time_frame_map = dict(TIMEFRAME_PROFILES[trading_strategy])

INSANE_SPIKE_SMOOTHING_CONFIG = {
    'price_columns': ('Open', 'High', 'Low', 'Close', 'Adj Close'),
    'rolling_window': 63,
    'absolute_log_return_threshold': np.log1p(0.35),
    'endpoint_absolute_log_return_threshold': np.log1p(0.75),
    'robust_z_threshold': 12.0,
    'endpoint_robust_z_threshold': 20.0,
    'interpolation_limit': 3,
}

def _robust_centered_mad_zscore(series, window):
    values = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    min_periods = max(10, int(window) // 4)
    rolling_median = values.rolling(window, center=True, min_periods=min_periods).median()
    rolling_mad = values.sub(rolling_median).abs().rolling(
        window,
        center=True,
        min_periods=min_periods,
    ).median()
    scale = (1.4826 * rolling_mad).replace(0, np.nan)
    return values.sub(rolling_median).div(scale).replace([np.inf, -np.inf], np.nan)

def _isolated_price_spike_mask(
    series,
    *,
    rolling_window=None,
    absolute_log_return_threshold=None,
    endpoint_absolute_log_return_threshold=None,
    robust_z_threshold=None,
    endpoint_robust_z_threshold=None,
):
    config = INSANE_SPIKE_SMOOTHING_CONFIG
    rolling_window = int(rolling_window or config['rolling_window'])
    absolute_log_return_threshold = float(
        absolute_log_return_threshold or config['absolute_log_return_threshold']
    )
    endpoint_absolute_log_return_threshold = float(
        endpoint_absolute_log_return_threshold
        or config['endpoint_absolute_log_return_threshold']
    )
    robust_z_threshold = float(robust_z_threshold or config['robust_z_threshold'])
    endpoint_robust_z_threshold = float(
        endpoint_robust_z_threshold or config['endpoint_robust_z_threshold']
    )

    clean = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    log_values = np.log(clean.where(clean > 0))
    inbound_return = log_values.diff()
    outbound_return = log_values.shift(-1).sub(log_values)
    inbound_z = _robust_centered_mad_zscore(inbound_return, rolling_window)
    outbound_z = _robust_centered_mad_zscore(outbound_return, rolling_window)

    inbound_extreme = (
        inbound_return.abs().gt(absolute_log_return_threshold)
        | inbound_z.abs().gt(robust_z_threshold)
    )
    outbound_extreme = (
        outbound_return.abs().gt(absolute_log_return_threshold)
        | outbound_z.abs().gt(robust_z_threshold)
    )
    isolated_reversal = (
        inbound_extreme
        & outbound_extreme
        & inbound_return.notna()
        & outbound_return.notna()
        & np.sign(inbound_return).ne(np.sign(outbound_return))
    )

    endpoint_spike = (
        outbound_return.isna()
        & inbound_return.notna()
        & (
            inbound_return.abs().gt(endpoint_absolute_log_return_threshold)
            | inbound_z.abs().gt(endpoint_robust_z_threshold)
        )
    )
    return (isolated_reversal | endpoint_spike).fillna(False)

def smooth_insane_price_spikes(
    price_frame,
    *,
    label=None,
    price_columns=None,
    rolling_window=None,
    absolute_log_return_threshold=None,
    endpoint_absolute_log_return_threshold=None,
    robust_z_threshold=None,
    endpoint_robust_z_threshold=None,
    interpolation_limit=None,
):
    if price_frame is None or price_frame.empty:
        return price_frame, pd.DataFrame(columns=['column', 'spike_count'])

    config = INSANE_SPIKE_SMOOTHING_CONFIG
    price_columns = tuple(price_columns or config['price_columns'])
    interpolation_limit = int(interpolation_limit or config['interpolation_limit'])
    smoothed = price_frame.copy()
    report_rows = []
    interpolation_method = 'time' if isinstance(smoothed.index, pd.DatetimeIndex) else 'linear'

    for column in price_columns:
        if column not in smoothed.columns:
            continue
        original = pd.to_numeric(smoothed[column], errors='coerce')
        spike_mask = _isolated_price_spike_mask(
            original,
            rolling_window=rolling_window,
            absolute_log_return_threshold=absolute_log_return_threshold,
            endpoint_absolute_log_return_threshold=endpoint_absolute_log_return_threshold,
            robust_z_threshold=robust_z_threshold,
            endpoint_robust_z_threshold=endpoint_robust_z_threshold,
        )
        if not bool(spike_mask.any()):
            continue

        cleaned = original.mask(spike_mask)
        cleaned = cleaned.interpolate(
            method=interpolation_method,
            limit=interpolation_limit,
            limit_direction='both',
        ).ffill().bfill()
        smoothed[column] = cleaned.where(original.notna(), original)
        report_rows.append({'column': column, 'spike_count': int(spike_mask.sum())})

    if {'Open', 'High', 'Low', 'Close'}.issubset(smoothed.columns):
        smoothed['High'] = pd.concat(
            [smoothed['High'], smoothed['Open'], smoothed['Close']],
            axis=1,
        ).max(axis=1)
        smoothed['Low'] = pd.concat(
            [smoothed['Low'], smoothed['Open'], smoothed['Close']],
            axis=1,
        ).min(axis=1)

    spike_report = pd.DataFrame(report_rows)
    if label and not spike_report.empty:
        total_spikes = int(spike_report['spike_count'].sum())
        adjusted_columns = ', '.join(spike_report['column'].astype(str))
        print(
            f'Smoothed {total_spikes} isolated price spike(s) in {label} '
            f'across: {adjusted_columns}'
        )
    return smoothed, spike_report

def smooth_insane_series_spikes(
    series,
    *,
    label=None,
    rolling_window=63,
    robust_z_threshold=8.0,
    interpolation_limit=5,
    floor=None,
    ceiling=None,
):
    values = pd.to_numeric(pd.Series(series).copy(), errors='coerce').replace([np.inf, -np.inf], np.nan)
    values = values.mask(values.lt(floor)) if floor is not None else values
    values = values.mask(values.gt(ceiling)) if ceiling is not None else values
    robust_z = _robust_centered_mad_zscore(values, int(rolling_window))
    spike_mask = robust_z.abs().gt(float(robust_z_threshold)).fillna(False)
    if ceiling is not None:
        spike_mask = spike_mask | pd.to_numeric(pd.Series(series), errors='coerce').gt(float(ceiling)).fillna(False)

    if not bool(spike_mask.any()):
        report = pd.DataFrame(columns=['series', 'spike_count'])
        values.name = getattr(series, 'name', None)
        return values, report

    interpolation_method = 'time' if isinstance(values.index, pd.DatetimeIndex) else 'linear'
    smoothed = values.mask(spike_mask).interpolate(
        method=interpolation_method,
        limit=int(interpolation_limit),
        limit_direction='both',
    ).ffill().bfill()
    if floor is not None:
        smoothed = smoothed.clip(lower=float(floor))
    if ceiling is not None:
        smoothed = smoothed.clip(upper=float(ceiling))
    smoothed.name = getattr(series, 'name', None)
    report = pd.DataFrame([
        {'series': label or getattr(series, 'name', 'series'), 'spike_count': int(spike_mask.sum())}
    ])
    if label:
        print(f'Smoothed {int(spike_mask.sum())} insane spike(s) in {label}.')
    return smoothed, report

market_history = get_market_history(
    symbols=[ticker_str],
    period=period,
    interval=interval,
    provider="yfinance",
    align=False,
)
ticker = market_history.get(str(ticker_str).strip().upper(), pd.DataFrame())
required_ohlc_columns = {"Open", "High", "Low", "Close"}
if ticker.empty or not required_ohlc_columns.issubset(ticker.columns):
    missing_columns = sorted(required_ohlc_columns.difference(ticker.columns))
    raise ValueError(
        f"No usable OHLC history returned for {ticker_str}; missing columns: {missing_columns}"
    )

ticker = ticker.loc[:, ["Open", "High", "Low", "Close"]].copy()
ticker.index = pd.to_datetime(ticker.index, errors="coerce", utc=True).tz_convert(None)
ticker = ticker.loc[~ticker.index.isna()].sort_index()
ticker, ticker_spike_smoothing_report = smooth_insane_price_spikes(
    ticker,
    label=ticker_str,
)

PLOTLY_NOTEBOOK_CONFIG = {"responsive": True, "scrollZoom": True}
for renderer_name in ("plotly_mimetype", "notebook", "notebook_connected", "jupyterlab"):
    try:
        pio.renderers[renderer_name].config = PLOTLY_NOTEBOOK_CONFIG.copy()
    except Exception:
        pass

def show_plotly_figure(fig, *, config=None, **layout_kwargs):
    merged_config = PLOTLY_NOTEBOOK_CONFIG.copy()
    if config:
        merged_config.update(config)
    fig.update_layout(autosize=True, **layout_kwargs)
    fig.show(config=merged_config)

print(
    f"Loaded {len(ticker):,} {interval} OHLC rows for {ticker_str} "
    f"({ticker.index.min():%Y-%m-%d} to {ticker.index.max():%Y-%m-%d})."
)


In [ ]:
# Block 2: Volatility Model Comparison

import sys

def _purge_stale_modules(prefixes):
    for prefix in prefixes:
        matching_modules = [
            name
            for name in list(sys.modules)
            if name == prefix or name.startswith(f"{prefix}.")
        ]
        for module_name in matching_modules:
            sys.modules.pop(module_name, None)

try:
    from arch import arch_model
except ModuleNotFoundError as exc:
    if exc.name != "arch":
        raise
    raise ImportError(
        "Volatility Model Comparison requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
    ) from exc
except Exception:
    _purge_stale_modules(("arch", "matplotlib"))
    try:
        from arch import arch_model
    except ModuleNotFoundError as exc:
        if exc.name != "arch":
            raise
        raise ImportError(
            "Volatility Model Comparison requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
        ) from exc
    except Exception as exc:
        raise RuntimeError(
            "Volatility Model Comparison could not import `arch` because the notebook kernel is holding a stale matplotlib state. Restart the kernel and rerun this notebook if this persists."
        ) from exc

close_series = ticker["Close"]
ohlc_frame = ticker[["Open", "High", "Low", "Close"]].copy()
returns = close_series.pct_change()
garch_input = returns.dropna() * 100

log_hl = np.log(ohlc_frame["High"] / ohlc_frame["Low"])
log_ho = np.log(ohlc_frame["High"] / ohlc_frame["Open"])
log_lo = np.log(ohlc_frame["Low"] / ohlc_frame["Open"])
log_co = np.log(ohlc_frame["Close"] / ohlc_frame["Open"])
log_oc = np.log(ohlc_frame["Open"] / ohlc_frame["Close"].shift(1))
log_hc = np.log(ohlc_frame["High"] / ohlc_frame["Close"])
log_lc = np.log(ohlc_frame["Low"] / ohlc_frame["Close"])

garman_klass_variance = 0.5 * (log_hl ** 2) - ((2 * np.log(2)) - 1) * (log_co ** 2)
parkinson_variance = (log_hl ** 2) / (4 * np.log(2))
rs_variance = (log_hc * log_ho) + (log_lc * log_lo)

volatility_model_specs = [
    ("GARCH(1,1)", dict(vol="GARCH", p=1, q=1, o=0), "#111111", "solid"),
    ("EGARCH(1,1)", dict(vol="EGARCH", p=1, o=1, q=1), "#d62728", "dash"),
    ("GJR-GARCH(1,1)", dict(vol="GARCH", p=1, o=1, q=1), "#2ca02c", "dot"),
]

rolling_realized_vol_specs = [
    ("Close-to-Close", "close-to-close", "#1f77b4", "solid"),
    ("Parkinson", "parkinson", "#9467bd", "dash"),
    ("Yang-Zhang", "yang-zhang", "#ff7f0e", "dot"),
    ("Garman-Klass", "garman-klass", "#8c564b", "dashdot"),
    ("Rogers-Satchell", "rogers-satchell", "#17becf", "longdash"),
]

ewma_realized_vol_specs = [
    ("EWMA Close-to-Close", "close-to-close", "#1f77b4", "longdashdot"),
    ("EWMA Parkinson", "parkinson", "#9467bd", "longdashdot"),
    ("EWMA Yang-Zhang", "yang-zhang", "#ff7f0e", "longdashdot"),
    ("EWMA Garman-Klass", "garman-klass", "#8c564b", "longdashdot"),
    ("EWMA Rogers-Satchell", "rogers-satchell", "#17becf", "longdashdot"),
]

volatility_series_spike_smoothing_reports = {}

def _smooth_block37_volatility_series(series, label, window=63):
    smoothed, report = smooth_insane_series_spikes(
        series,
        label=None,
        rolling_window=max(21, min(126, int(window) * 3)),
        robust_z_threshold=8.0,
        interpolation_limit=max(3, min(10, int(window) // 2)),
        floor=0.0,
    )
    if not report.empty:
        volatility_series_spike_smoothing_reports[label] = report
    smoothed.name = getattr(series, 'name', label)
    return smoothed

annualized_model_vols_raw = {}
annualized_model_vols = {}
for model_name, model_kwargs, _, _ in volatility_model_specs:
    model_fit = arch_model(
        garch_input,
        mean="Zero",
        dist="normal",
        rescale=False,
        **model_kwargs,
    ).fit(disp="off")
    annualized_model_vol = (model_fit.conditional_volatility / 100.0) * np.sqrt(252)
    annualized_model_vol.name = f"Annualized {model_name}"
    annualized_model_vols_raw[model_name] = annualized_model_vol.copy()
    annualized_model_vol = _smooth_block37_volatility_series(
        annualized_model_vol,
        f'{model_name} conditional volatility',
        window=63,
    )
    annualized_model_vol.name = f"Annualized {model_name}"
    annualized_model_vols[model_name] = annualized_model_vol

def compute_rolling_realized_vol_series(window, method):
    if method == "close-to-close":
        raw_series = returns.rolling(window).std() * np.sqrt(252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Close-to-Close rolling realized vol ({window})',
            window=window,
        )
    if method == "parkinson":
        rolling_variance = parkinson_variance.rolling(window=window).mean().clip(lower=0)
    elif method == "yang-zhang":
        if window < 2:
            return pd.Series(np.nan, index=ohlc_frame.index)
        k = 0.34 / (1.34 + ((window + 1) / (window - 1)))
        overnight_variance = log_oc.rolling(window=window).var()
        open_to_close_variance = log_co.rolling(window=window).var()
        rs_component = rs_variance.rolling(window=window).mean()
        rolling_variance = (
            overnight_variance
            + (k * open_to_close_variance)
            + ((1 - k) * rs_component)
        ).clip(lower=0)
    elif method == "garman-klass":
        rolling_variance = garman_klass_variance.rolling(window=window).mean().clip(lower=0)
    elif method == "rogers-satchell":
        rolling_variance = rs_variance.rolling(window=window).mean().clip(lower=0)
    else:
        raise ValueError(f"Unsupported rolling volatility method: {method}")

    raw_series = np.sqrt(rolling_variance) * np.sqrt(252)
    return _smooth_block37_volatility_series(
        raw_series,
        f'{method} rolling realized vol ({window})',
        window=window,
    )

def compute_ewma_realized_vol_series(window, method):
    alpha = 2.0 / (window + 1.0)

    if method == "close-to-close":
        ewma_variance = returns.pow(2).ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        raw_series = np.sqrt(ewma_variance * 252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Close-to-Close EWMA realized vol ({window})',
            window=window,
        )

    if method == "parkinson":
        ewma_variance = parkinson_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        raw_series = np.sqrt(ewma_variance * 252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Parkinson EWMA realized vol ({window})',
            window=window,
        )

    if method == "garman-klass":
        ewma_variance = garman_klass_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        raw_series = np.sqrt(ewma_variance * 252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Garman-Klass EWMA realized vol ({window})',
            window=window,
        )

    if method == "rogers-satchell":
        ewma_variance = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        raw_series = np.sqrt(ewma_variance * 252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Rogers-Satchell EWMA realized vol ({window})',
            window=window,
        )

    if method == "yang-zhang":
        if window < 2:
            return pd.Series(np.nan, index=ohlc_frame.index)

        k = 0.34 / (1.34 + ((window + 1) / (window - 1)))
        overnight_variance = log_oc.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        open_to_close_variance = log_co.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        rs_component = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        yz_variance = (
            overnight_variance
            + (k * open_to_close_variance)
            + ((1 - k) * rs_component)
        ).clip(lower=0)
        raw_series = np.sqrt(yz_variance * 252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Yang-Zhang EWMA realized vol ({window})',
            window=window,
        )

    raise ValueError(f"Unsupported EWMA volatility method: {method}")

volatility_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
default_vol_term = 'long' if 'long' in volatility_term_order else max(
    volatility_term_order,
    key=lambda term: int(time_frame_map[term]),
)

volatility_term_plot_data = {}
for term in volatility_term_order:
    window = int(time_frame_map[term])
    rolling_realized_vol_map = {
        label: compute_rolling_realized_vol_series(window, method)
        for label, method, _, _ in rolling_realized_vol_specs
    }
    ewma_realized_vol_map = {
        label: compute_ewma_realized_vol_series(window, method)
        for label, method, _, _ in ewma_realized_vol_specs
    }

    non_empty_series = [
        series
        for series in (
            [series.dropna() for series in rolling_realized_vol_map.values()]
            + [series.dropna() for series in ewma_realized_vol_map.values()]
            + [model_series.dropna() for model_series in annualized_model_vols.values()]
        )
        if not series.empty
    ]
    if non_empty_series:
        max_index = max(series.index.max() for series in non_empty_series)
        min_index = min(series.index.min() for series in non_empty_series)
        term_range = [max(min_index, max_index - pd.DateOffset(years=3)), max_index]
    else:
        term_range = None

    volatility_term_plot_data[term] = {
        'window': window,
        'rolling_realized_vol_map': rolling_realized_vol_map,
        'ewma_realized_vol_map': ewma_realized_vol_map,
        'term_range': term_range,
    }

if volatility_series_spike_smoothing_reports:
    total_smoothed_vol_spikes = int(
        sum(report['spike_count'].sum() for report in volatility_series_spike_smoothing_reports.values())
    )
    print(
        f'Smoothed {total_smoothed_vol_spikes} insane volatility-model/realized-vol spike(s). '
        'See volatility_series_spike_smoothing_reports for details.'
    )

vol_model_fig = plot_volatility_model_comparison_view(
    annualized_model_vols=annualized_model_vols,
    term_plot_data=volatility_term_plot_data,
    volatility_model_specs=volatility_model_specs,
    rolling_realized_vol_specs=rolling_realized_vol_specs,
    ewma_realized_vol_specs=ewma_realized_vol_specs,
    term_order=volatility_term_order,
    default_term=default_vol_term,
    time_frame_map=time_frame_map,
    ticker_label=ticker_str,
)
show_plotly_figure(vol_model_fig)

In [ ]:
# Block 3: Range-Based Sharpe Ratio Comparison

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

TRADING_DAYS_PER_YEAR = 252
range_sharpe_colors = {
    "Standard Sharpe": "#f8fafc",
    "Parkinson Sharpe": "#a78bfa",
    "Garman-Klass Sharpe": "#f59e0b",
    "Yang-Zhang Sharpe": "#22d3ee",
    "EWMA Standard Sharpe": "#f8fafc",
    "EWMA Parkinson Sharpe": "#a78bfa",
    "EWMA Garman-Klass Sharpe": "#f59e0b",
    "EWMA Yang-Zhang Sharpe": "#22d3ee",
}
range_sharpe_dashes = {
    "Standard Sharpe": "solid",
    "Parkinson Sharpe": "dash",
    "Garman-Klass Sharpe": "dot",
    "Yang-Zhang Sharpe": "dashdot",
    "EWMA Standard Sharpe": "longdashdot",
    "EWMA Parkinson Sharpe": "longdashdot",
    "EWMA Garman-Klass Sharpe": "longdashdot",
    "EWMA Yang-Zhang Sharpe": "longdashdot",
}

# Use the prior session's annualized Treasury-bill yield to avoid look-ahead.
risk_free_ticker = "^IRX"
risk_free_history = get_market_history(
    symbols=[risk_free_ticker],
    period=period,
    interval=interval,
    provider="yfinance",
    align=False,
).get(risk_free_ticker, pd.DataFrame())
if isinstance(risk_free_history, pd.DataFrame) and "Close" in risk_free_history:
    annual_risk_free_yield = (
        pd.to_numeric(risk_free_history["Close"], errors="coerce")
        .sort_index()
        .div(100.0)
        .reindex(ticker.index)
        .ffill()
    )
    daily_risk_free_rate = (
        (1.0 + annual_risk_free_yield).pow(1.0 / TRADING_DAYS_PER_YEAR).sub(1.0)
        .shift(1)
        .fillna(0.0)
    )
else:
    daily_risk_free_rate = pd.Series(0.0, index=ticker.index)
    print(f"{risk_free_ticker} was unavailable; using a 0% risk-free-rate fallback.")

excess_returns = returns.sub(daily_risk_free_rate, fill_value=0.0)

def compute_range_based_sharpe(window):
    """Compare Sharpe ratios while changing only the volatility estimator."""
    window = int(window)
    if window < 2:
        raise ValueError("The Sharpe comparison window must be at least 2 sessions.")

    annualized_excess_return = (
        excess_returns.rolling(window, min_periods=window).mean()
        * TRADING_DAYS_PER_YEAR
    )
    standard_volatility = (
        excess_returns.rolling(window, min_periods=window).std()
        * np.sqrt(TRADING_DAYS_PER_YEAR)
    )
    parkinson_volatility = np.sqrt(
        parkinson_variance.rolling(window, min_periods=window).mean().clip(lower=0.0)
        * TRADING_DAYS_PER_YEAR
    )
    garman_klass_volatility = np.sqrt(
        garman_klass_variance.rolling(window, min_periods=window).mean().clip(lower=0.0)
        * TRADING_DAYS_PER_YEAR
    )

    yang_zhang_k = 0.34 / (1.34 + ((window + 1.0) / (window - 1.0)))
    yang_zhang_variance = (
        log_oc.rolling(window, min_periods=window).var()
        + yang_zhang_k * log_co.rolling(window, min_periods=window).var()
        + (1.0 - yang_zhang_k)
        * rs_variance.rolling(window, min_periods=window).mean()
    ).clip(lower=0.0)
    yang_zhang_volatility = np.sqrt(
        yang_zhang_variance * TRADING_DAYS_PER_YEAR
    )

    ewma_alpha = 2.0 / (window + 1.0)
    ewma_standard_volatility = (
        excess_returns.ewm(
            alpha=ewma_alpha, adjust=False, min_periods=window
        ).std(bias=False)
        * np.sqrt(TRADING_DAYS_PER_YEAR)
    )
    ewma_parkinson_volatility = np.sqrt(
        parkinson_variance.ewm(
            alpha=ewma_alpha, adjust=False, min_periods=window
        ).mean().clip(lower=0.0)
        * TRADING_DAYS_PER_YEAR
    )
    ewma_garman_klass_volatility = np.sqrt(
        garman_klass_variance.ewm(
            alpha=ewma_alpha, adjust=False, min_periods=window
        ).mean().clip(lower=0.0)
        * TRADING_DAYS_PER_YEAR
    )
    ewma_yang_zhang_variance = (
        log_oc.ewm(
            alpha=ewma_alpha, adjust=False, min_periods=window
        ).var(bias=False)
        + yang_zhang_k * log_co.ewm(
            alpha=ewma_alpha, adjust=False, min_periods=window
        ).var(bias=False)
        + (1.0 - yang_zhang_k) * rs_variance.ewm(
            alpha=ewma_alpha, adjust=False, min_periods=window
        ).mean()
    ).clip(lower=0.0)
    ewma_yang_zhang_volatility = np.sqrt(
        ewma_yang_zhang_variance * TRADING_DAYS_PER_YEAR
    )

    volatility_frame = pd.concat(
        {
            "Standard Sharpe": standard_volatility,
            "Parkinson Sharpe": parkinson_volatility,
            "Garman-Klass Sharpe": garman_klass_volatility,
            "Yang-Zhang Sharpe": yang_zhang_volatility,
            "EWMA Standard Sharpe": ewma_standard_volatility,
            "EWMA Parkinson Sharpe": ewma_parkinson_volatility,
            "EWMA Garman-Klass Sharpe": ewma_garman_klass_volatility,
            "EWMA Yang-Zhang Sharpe": ewma_yang_zhang_volatility,
        },
        axis=1,
    )
    volatility_frame = volatility_frame.where(volatility_frame > 0.0)
    ratio_frame = volatility_frame.rdiv(annualized_excess_return, axis=0)
    ratio_frame = ratio_frame.replace([np.inf, -np.inf], np.nan)
    ratio_frame.index.name = "Date"
    return ratio_frame, volatility_frame

range_sharpe_by_term = {}
range_sharpe_zscore_by_term = {}
range_sharpe_volatility_by_term = {}
for term in volatility_term_order:
    term_window = int(time_frame_map[term])
    ratio_frame, denominator_frame = compute_range_based_sharpe(term_window)
    range_sharpe_by_term[term] = ratio_frame
    minimum_zscore_observations = max(20, term_window)
    expanding_mean = ratio_frame.expanding(
        min_periods=minimum_zscore_observations
    ).mean()
    expanding_std = ratio_frame.expanding(
        min_periods=minimum_zscore_observations
    ).std(ddof=1)
    range_sharpe_zscore_by_term[term] = (
        ratio_frame.sub(expanding_mean).div(expanding_std.where(expanding_std > 0.0))
    ).replace([np.inf, -np.inf], np.nan)
    range_sharpe_volatility_by_term[term] = denominator_frame

range_sharpe_summary_rows = []
for term in volatility_term_order:
    term_window = int(time_frame_map[term])
    ratio_frame = range_sharpe_by_term[term]
    standard_series = ratio_frame["Standard Sharpe"]
    for model_name in ratio_frame.columns:
        model_series = ratio_frame[model_name].dropna()
        latest_value = float(model_series.iloc[-1]) if not model_series.empty else np.nan
        latest_date = model_series.index[-1] if not model_series.empty else pd.NaT
        standard_value = standard_series.reindex([latest_date]).iloc[0] if pd.notna(latest_date) else np.nan
        range_sharpe_summary_rows.append(
            {
                "Horizon": term.title(),
                "Window": term_window,
                "Ratio": model_name,
                "Latest": latest_value,
                "Difference vs Standard": latest_value - standard_value,
                "As Of": latest_date,
            }
        )
range_sharpe_summary = pd.DataFrame(range_sharpe_summary_rows)

def build_range_sharpe_comparison_figure():
    figure = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.065,
        row_heights=[0.48, 0.24, 0.28],
        subplot_titles=(
            "Rolling Sharpe Ratio by Volatility Estimator",
            "Range-Based Sharpe Minus Standard Sharpe",
            "Expanding Historical Z-Score by Volatility Estimator",
        ),
    )
    trace_term_membership = []
    default_term = default_vol_term

    for term in volatility_term_order:
        term_window = int(time_frame_map[term])
        ratio_frame = range_sharpe_by_term[term]
        is_visible = term == default_term
        for model_name in range_sharpe_colors:
            model_series = ratio_frame[model_name]
            figure.add_trace(
                go.Scatter(
                    x=model_series.index,
                    y=model_series,
                    mode="lines",
                    name=model_name,
                    legendgroup=model_name,
                    line=dict(
                        color=range_sharpe_colors[model_name],
                        width=2.4 if model_name == "Standard Sharpe" else 1.9,
                        dash=range_sharpe_dashes[model_name],
                    ),
                    visible=is_visible,
                    showlegend=True,
                    hovertemplate=(
                        f"Estimator: {model_name}<br>"
                        f"Window: {term_window} sessions<br>"
                        "Date: %{x|%Y-%m-%d}<br>"
                        "Sharpe ratio: %{y:.3f}<extra></extra>"
                    ),
                ),
                row=1,
                col=1,
            )
            trace_term_membership.append(term)

        for model_name in (
            name for name in range_sharpe_colors if name != "Standard Sharpe"
        ):
            spread = ratio_frame[model_name] - ratio_frame["Standard Sharpe"]
            figure.add_trace(
                go.Scatter(
                    x=spread.index,
                    y=spread,
                    mode="lines",
                    name=f"{model_name} - Standard",
                    legendgroup=model_name,
                    line=dict(
                        color=range_sharpe_colors[model_name],
                        width=1.6,
                        dash=range_sharpe_dashes[model_name],
                    ),
                    visible=is_visible,
                    showlegend=False,
                    hovertemplate=(
                        f"Estimator: {model_name}<br>"
                        f"Window: {term_window} sessions<br>"
                        "Date: %{x|%Y-%m-%d}<br>"
                        "Difference vs standard: %{y:+.3f}<extra></extra>"
                    ),
                ),
                row=2,
                col=1,
            )
            trace_term_membership.append(term)

        zscore_frame = range_sharpe_zscore_by_term[term]
        for model_name in range_sharpe_colors:
            zscore_series = zscore_frame[model_name]
            figure.add_trace(
                go.Scatter(
                    x=zscore_series.index,
                    y=zscore_series,
                    mode="lines",
                    name=f"{model_name} Z-Score",
                    legendgroup=model_name,
                    line=dict(
                        color=range_sharpe_colors[model_name],
                        width=2.1 if model_name == "Standard Sharpe" else 1.7,
                        dash=range_sharpe_dashes[model_name],
                    ),
                    visible=is_visible,
                    showlegend=False,
                    hovertemplate=(
                        f"Estimator: {model_name}<br>"
                        f"Window: {term_window} sessions<br>"
                        "Date: %{x|%Y-%m-%d}<br>"
                        "Historical z-score: %{y:.3f}<extra></extra>"
                    ),
                ),
                row=3,
                col=1,
            )
            trace_term_membership.append(term)

    term_buttons = []
    for term in volatility_term_order:
        term_window = int(time_frame_map[term])
        term_buttons.append(
            dict(
                label=f"{term.title()} ({term_window}d)",
                method="update",
                args=[
                    {"visible": [owner_term == term for owner_term in trace_term_membership]},
                    {
                        "title.text": (
                            f"{ticker_str} Range-Based Sharpe Comparison "
                            f"[{term_window}-Session Window]"
                        )
                    },
                ],
            )
        )

    all_dates = pd.DatetimeIndex(
        sorted(
            set().union(
                *(frame.dropna(how="all").index for frame in range_sharpe_by_term.values())
            )
        )
    )
    default_date_range = None
    if not all_dates.empty:
        default_end = all_dates.max()
        default_start = max(all_dates.min(), default_end - pd.DateOffset(years=3))
        default_date_range = [default_start, default_end]

    default_window = int(time_frame_map[default_term])
    figure.add_hline(
        y=0.0, line_color="rgba(226,232,240,0.55)", line_width=1.2,
        row=1, col=1,
    )
    figure.add_hline(
        y=0.0, line_color="rgba(226,232,240,0.65)", line_width=1.2,
        row=2, col=1,
    )
    figure.add_hline(
        y=0.0, line_color="rgba(226,232,240,0.65)", line_width=1.2,
        row=3, col=1,
    )
    figure.update_yaxes(title_text="Annualized ratio", row=1, col=1)
    figure.update_yaxes(title_text="Difference", row=2, col=1)
    figure.update_yaxes(title_text="Historical z-score", row=3, col=1)
    figure.update_xaxes(title_text="Date", row=3, col=1)
    if default_date_range is not None:
        figure.update_xaxes(range=default_date_range)
    figure.update_layout(
        title=f"{ticker_str} Range-Based Sharpe Comparison [{default_window}-Session Window]",
        template="plotly_dark",
        height=1260,
        hovermode="x unified",
        font=dict(
            family="Inter, Aptos, Segoe UI, Arial, sans-serif",
            color="#e2e8f0",
        ),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0,
        ),
        updatemenus=[
            dict(
                type="dropdown",
                direction="down",
                buttons=term_buttons,
                active=volatility_term_order.index(default_term),
                x=0.0,
                y=1.16,
                xanchor="left",
                yanchor="top",
                bgcolor="#111827",
                bordercolor="#475569",
                font=dict(color="#f8fafc"),
            )
        ],
        margin=dict(t=135, r=35, b=55, l=75),
    )
    return figure

range_sharpe_fig = build_range_sharpe_comparison_figure()
display(
    range_sharpe_summary.style.format(
        {
            "Latest": "{:.3f}",
            "Difference vs Standard": "{:+.3f}",
            "As Of": lambda value: value.strftime("%Y-%m-%d") if pd.notna(value) else "N/A",
        }
    )
)
show_plotly_figure(range_sharpe_fig)
